# Run the existing complete pipeline
Use the configured rttpc Jupyter kernel. Run top to bottom after pulling the code.
All model loading, depth, ground extraction, SAM, CLIP, mapping, and plotting
remain inside `pipeline()`. Only the native Open3D window is disabled.

## 1. Find the repository and enable inline plots
Works from the checkout root or its notebooks folder. Restart the kernel after
pulling changes if an older pipeline module was already imported.


In [ ]:
%matplotlib inline
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = next(
    (p for p in (cwd, *cwd.parents) if (p / "src" / "rttpc" / "pipeline.py").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Open this notebook from the RTTPC root or notebooks folder.")
sys.path.insert(0, str(repo_root / "src"))
from rttpc.pipeline import pipeline
print(f"Repository: {repo_root}")


## 2. Validate the original RELLIS input
The only dataset file read by the current pipeline is:
`Rellis_3D_image_example/pylon_camera_node/frame000000-1581623790_349.jpg`.

For a checkout at /home/jovyan/RTTPC, place that exact image at
`/home/jovyan/RTTPC/Rellis_3D_image_example/pylon_camera_node/frame000000-1581623790_349.jpg`.
The dataset directory is ignored by Git and must already be available on the server.
No LiDAR scans, labels, pose files, or calibration files are read.

Calibration remains hardcoded in the pipeline:
fx/fy/cx/cy = [2813.643275, 2808.326079, 969.285772, 624.049972];
camera-to-lidar quaternion (w,x,y,z) =
[-0.50507811, 0.51206185, 0.49024953, -0.49228464];
translation = [-0.13165462, 0.03870398, -0.17253834].
Use the original image at its original size. Edit the path only to locate this
same file elsewhere; this notebook supplies no replacement image or calibration.


In [ ]:
IMAGE_PATH = repo_root / "Rellis_3D_image_example/pylon_camera_node/frame000000-1581623790_349.jpg"
if not IMAGE_PATH.is_file():
    raise FileNotFoundError(
        f"Required RELLIS image missing: {IMAGE_PATH}. "
        "Make the original file available on the server before running."
    )
print(f"Input: {IMAGE_PATH}")


## 3. Call the complete pipeline
This loads the existing SAM ViT-base, CLIP ViT-base-patch32, and metric outdoor
Depth Anything V2 Small models. Weights need to be cached or accessible.
The original two mask overlays and map plot appear inline. The returned
dictionary retains intermediates for inspection without recomputing them.

Existing geometry/map issues are deliberately preserved: the camera-to-lidar
transform is applied twice; robot-to-world conversion is unfinished; the map
starts at (48,48) metres and indices are unchecked. Out-of-range indices may
wrap or raise an error. A completed run does not validate map correctness.
Invalid CLIP scores now stop clearly, and zero masks do not divide by zero.


In [ ]:
results = pipeline(image_path=IMAGE_PATH, show_native_gui=False)
print(f"Accepted masks: {len(results['segmentations'])}")
print(f"Ground points: {len(results['ground_indices'])}")
print(f"Map shape: {results['trav_map'].data.shape}")
print(f"Available results: {', '.join(results)}")
